# Cell 1 — Dependency Installation

In [ ]:
"""
PersianScribe — A Whisper-based Persian Audio-to-Text Toolkit.

This cell bootstraps the runtime by installing required Python packages.
On Colab, simply run this cell. On local Windows/Linux machines, ensure
`ffmpeg` is installed and available on the system PATH.
"""

import shutil
import subprocess
import sys


def pip_install(*packages: str) -> None:
    """Install one or more packages via pip, streaming output to stdout.

    Args:
        *packages: PyPI package specifiers or pip flags (e.g. "--index-url ...").

    Raises:
        RuntimeError: If the underlying pip invocation exits with a non-zero code.
    """
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *packages]
    print(">>>", " ".join(cmd))
    result = subprocess.run(
        cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"pip failed while installing: {packages}")


# Core libraries required for transcription, progress bars, widgets, and DOCX export.
pip_install("openai-whisper", "tqdm", "ipywidgets", "python-docx")

# --- Optional: CUDA-enabled PyTorch -----------------------------------------
# If you own an NVIDIA GPU with CUDA 12.x, uncomment the block below to install
# the matching wheel. Otherwise the CPU build pulled in by openai-whisper is used.
#
# try:
#     pip_install("torch", "--index-url", "https://download.pytorch.org/whl/cu121")
# except RuntimeError:
#     print("CUDA wheel install failed — falling back to CPU-only torch.")
#     pip_install("torch")

# --- Verify ffmpeg availability ---------------------------------------------
print("\nChecking for ffmpeg...")
ffmpeg_available = shutil.which("ffmpeg") is not None
print(f"ffmpeg found: {ffmpeg_available}")
if not ffmpeg_available:
    print(
        "⚠️  ffmpeg was not found on PATH.\n"
        "    • Colab : run `!apt-get -y install ffmpeg`\n"
        "    • Windows: download from https://ffmpeg.org and add it to PATH"
    )

# Cell 2 — Core Engine

In [ ]:
"""
Core transcription engine.

Defines the configuration schema, output writers (TXT/SRT/DOCX), and the
`WhisperTranscriber` class that wraps model loading and inference.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import timedelta
from pathlib import Path
from typing import Any, Optional

import torch
import whisper

# Audio/video container extensions accepted by the pipeline.
SUPPORTED_EXTENSIONS: frozenset[str] = frozenset(
    {".m4a", ".mp3", ".wav", ".flac", ".mkv", ".mp4", ".aac", ".ogg", ".wma", ".m4b"}
)


@dataclass(slots=True)
class TranscriptionConfig:
    """User-facing configuration for a transcription run.

    Attributes:
        model_size:        Whisper checkpoint size: tiny | base | small | medium | large.
        device_preference: Compute device preference: auto | cuda | cpu.
        language:          ISO-639 language code of the audio (e.g. "fa" for Persian).
        task:              "transcribe" (same language) or "translate" (to English).
        make_docx:         Whether to additionally export a formatted DOCX file.
        output_dir:        Directory where TXT/SRT/DOCX outputs are written.
    """

    model_size: str = "medium"
    device_preference: str = "auto"
    language: str = "fa"
    task: str = "transcribe"
    make_docx: bool = False
    output_dir: Path = field(default_factory=lambda: Path("./outputs"))


def format_timestamp(seconds: float) -> str:
    """Convert a duration in seconds to an SRT timestamp (HH:MM:SS,mmm)."""
    total_ms = int(timedelta(seconds=seconds).total_seconds() * 1000)
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1_000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def write_srt(segments: list[dict[str, Any]], srt_path: Path) -> None:
    """Write Whisper segments to a SubRip (.srt) subtitle file."""
    srt_path.parent.mkdir(parents=True, exist_ok=True)
    with srt_path.open("w", encoding="utf-8") as handle:
        for index, segment in enumerate(segments, start=1):
            start = format_timestamp(segment["start"])
            end = format_timestamp(segment["end"])
            text = segment["text"].strip()
            handle.write(f"{index}\n{start} --> {end}\n{text}\n\n")


def write_txt(segments: list[dict[str, Any]], txt_path: Path) -> None:
    """Write the plain-text transcript, one segment per line."""
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    with txt_path.open("w", encoding="utf-8") as handle:
        for segment in segments:
            handle.write(segment["text"].strip() + "\n")


def write_docx(
    segments: list[dict[str, Any]],
    docx_path: Path,
    include_timestamps: bool = True,
) -> None:
    """Export the transcript to a formatted Word (.docx) document.

    Args:
        segments:           Whisper transcription segments.
        docx_path:          Destination path for the document.
        include_timestamps: Prefix each paragraph with its time range when True.
    """
    from docx import Document
    from docx.shared import Pt

    document = Document()
    normal_style = document.styles["Normal"]
    normal_style.font.name = "Calibri"
    normal_style.font.size = Pt(11)

    document.add_heading("Transcript", level=1)
    for segment in segments:
        text = segment["text"].strip()
        prefix = ""
        if include_timestamps:
            prefix = (
                f"[{format_timestamp(segment['start'])} – "
                f"{format_timestamp(segment['end'])}] "
            )
        document.add_paragraph(prefix + text)

    document.save(str(docx_path))


@dataclass
class TranscriptionResult:
    """Bundles the artifacts produced for a single audio file."""

    txt_path: Path
    srt_path: Path
    docx_path: Optional[Path]
    segments: list[dict[str, Any]]


class WhisperTranscriber:
    """High-level wrapper around an OpenAI Whisper model.

    Handles device resolution, lazy model loading, and per-file transcription
    with automatic export to TXT/SRT (and optionally DOCX).
    """

    def __init__(self, config: TranscriptionConfig) -> None:
        self.config = config
        self.device, self.use_fp16 = self._resolve_device(config.device_preference)
        self._model: Optional[whisper.Whisper] = None

    @staticmethod
    def _resolve_device(preference: str) -> tuple[str, bool]:
        """Resolve the effective device and whether FP16 inference is enabled.

        FP16 is only enabled on CUDA devices, as CPU FP16 is unsupported by Whisper.
        """
        cuda_available = torch.cuda.is_available()
        if preference == "cpu":
            return "cpu", False
        if preference == "cuda" and not cuda_available:
            print("⚠️  CUDA requested but unavailable — falling back to CPU.")
        device = "cuda" if (preference != "cpu" and cuda_available) else "cpu"
        return device, device == "cuda"

    def load(self) -> None:
        """Load the Whisper checkpoint onto the resolved device (idempotent)."""
        if self._model is not None:
            return
        print(
            f"Loading Whisper model '{self.config.model_size}' "
            f"on {self.device} (fp16={self.use_fp16}) ..."
        )
        self._model = whisper.load_model(self.config.model_size, device=self.device)
        print("Model loaded successfully.")

    @property
    def model(self) -> whisper.Whisper:
        """Return the loaded model, loading it on first access if needed."""
        if self._model is None:
            self.load()
        assert self._model is not None
        return self._model

    def transcribe(self, audio_path: Path) -> TranscriptionResult:
        """Transcribe a single audio file and persist its outputs.

        Args:
            audio_path: Path to the source audio/video file.

        Returns:
            A TranscriptionResult describing every generated artifact.
        """
        result = self.model.transcribe(
            str(audio_path),
            language=self.config.language,
            task=self.config.task,
            verbose=False,
            fp16=self.use_fp16,
        )
        segments: list[dict[str, Any]] = result.get("segments", [])

        stem = audio_path.stem
        txt_path = self.config.output_dir / f"{stem}.txt"
        srt_path = self.config.output_dir / f"{stem}.srt"
        write_txt(segments, txt_path)
        write_srt(segments, srt_path)

        docx_path: Optional[Path] = None
        if self.config.make_docx:
            docx_path = self.config.output_dir / f"{stem}.docx"
            write_docx(segments, docx_path, include_timestamps=True)

        return TranscriptionResult(txt_path, srt_path, docx_path, segments)

# Cell 3 — Configuration & Model Loading

In [ ]:
"""Instantiate the transcriber with your preferred settings."""

config = TranscriptionConfig(
    model_size="medium",       # tiny | base | small | medium | large
    device_preference="cuda",  # auto | cuda | cpu
    language="fa",             # "fa" = Persian
    task="transcribe",         # "transcribe" | "translate" (-> English)
    make_docx=False,           # Set True to also export a DOCX file
    output_dir=Path("./outputs"),
)

transcriber = WhisperTranscriber(config)
transcriber.load()

# Cell 4 — Single-File Mode

In [ ]:
"""Transcribe a single file given its path."""

single_audio_path = r""

if single_audio_path:
    audio_path = Path(single_audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"File not found: {audio_path}")

    result = transcriber.transcribe(audio_path)

    artifacts = [str(result.txt_path), str(result.srt_path)]
    if result.docx_path:
        artifacts.append(str(result.docx_path))
    print("Saved:", " | ".join(artifacts))

    preview = "".join(seg.get("text", "") for seg in result.segments)[:2000]
    print("\n=== Preview (first ~2000 chars) ===\n", preview)
else:
    print("⚠️  Set `single_audio_path`, or use the upload/batch cells below.")

# Cell 5 — Widget Upload Mode (Multiple Files)

In [ ]:
"""Upload and transcribe one or more files directly from the notebook UI."""

import ipywidgets as widgets
from IPython.display import display

UPLOAD_DIR = Path("./_uploaded")

upload_widget = widgets.FileUpload(accept="audio/*,video/*", multiple=True)
transcribe_button = widgets.Button(
    description="Transcribe Uploaded Files", button_style="primary"
)
output_area = widgets.Output()


def _handle_transcribe_click(_button: widgets.Button) -> None:
    """Persist uploaded files to disk and run transcription on each."""
    with output_area:
        output_area.clear_output()

        # ipywidgets >= 8 exposes `.value` as a tuple of file dicts.
        uploaded_files = upload_widget.value
        if not uploaded_files:
            print("No files uploaded.")
            return

        UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
        for file_info in uploaded_files:
            name = file_info["name"]
            temp_path = UPLOAD_DIR / name
            temp_path.write_bytes(file_info["content"])

            if temp_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                print(f"Skipping unsupported file: {name}")
                continue

            print(f"Transcribing: {name}")
            try:
                result = transcriber.transcribe(temp_path)
                names = [result.txt_path.name, result.srt_path.name]
                if result.docx_path:
                    names.append(result.docx_path.name)
                print("  Saved:", " | ".join(names))
            except Exception as exc:  # noqa: BLE001 — surface any failure to the UI
                print(f"  [ERROR] {name}: {exc}")


transcribe_button.on_click(_handle_transcribe_click)
display(upload_widget, transcribe_button, output_area)

# Cell 6 — Batch Folder Mode

In [ ]:
"""Recursively transcribe every supported audio file inside a folder."""

from tqdm.auto import tqdm

batch_input_dir = r""

if batch_input_dir:
    input_dir = Path(batch_input_dir)
    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder does not exist: {input_dir}")

    audio_files = sorted(
        path
        for path in input_dir.rglob("*")
        if path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    print(f"Found {len(audio_files)} audio file(s).")

    for audio_path in tqdm(audio_files, desc="Batch", unit="file"):
        try:
            transcriber.transcribe(audio_path)
        except Exception as exc:  # noqa: BLE001 — keep the batch running on errors
            print(f"[ERROR] {audio_path.name}: {exc}")

    print("Done. Outputs are in:", config.output_dir.resolve())
else:
    print("⚠️  Set `batch_input_dir` to enable batch processing.")